# NNLM (Neural Network Language Model)

신경망 기반 언어 모델인 NNLM을 학습한다.

n-gram 언어 모델은 단어 조합의 빈도를 세어서 다음 단어를 예측했다. 이 방식은 직관적이지만 한계가 있다.

예를 들어 `I love you`와 `I like you`는 의미적으로 비슷하지만, n-gram 모델은 `love`와 `like`가 비슷한 단어라는 사실을 알지 못한다. 단어를 단순한 문자열로만 보기 때문이다.

NNLM은 이 문제를 줄이기 위해 단어를 숫자 벡터, 즉 임베딩으로 바꾼다. 그리고 신경망이 이 임베딩 벡터를 사용해 다음 단어를 예측한다.

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(42)

In [6]:
# 예제 말뭉치
# I love you / I love deep learning NLP / I love NLP 를 단순화한 데이터
word_to_index = {
    'I' : 0,
    'love' : 1,
    'you' : 2,
    'deep' : 3,
    'learning' : 4,
    'NLP' : 5,
    '<UNK>' : 6
}

index_to_word = {v:k for k,v in word_to_index.items()}
vocab_size = len(word_to_index)
embedding_dim = 5
hidden_dim = 10
# 이전 단어 2개를 보고 다음 단어를 예측
context_size = 2

In [7]:
# (입력 문맥 2단어, 다음 단어) 형태의 학습 데이터
training_data = [
    (['I','love'],'you'),
    (['I','love'],'deep'),
    (['love','deep'],'learning'),
    (['deep','learning'],'NLP'),
    (['I','love'],'NLP'),
]

In [8]:
def encode(words,word_to_index):
    """
    단어 리스트를 정수 id tensor로 변환하는 정수
    """
    unk = word_to_index['<UNK>']
    indices = [word_to_index.get(word,unk) for word in words]

    return torch.tensor(indices,dtype=torch.long)

# 변환 예시
sample = encode(['I','love','NLP'],word_to_index)
print(sample)

tensor([0, 1, 5])


In [9]:
# NNLM 모델 정의
class NNLM(nn.Module):
    def __init__(self, vocab_size,embedding_dim,context_size,hidden_dim):
        super().__init__()

        # 단어 id를 임베딩 벡터로 변환하는 중
        self.embedding = nn.Embedding(vocab_size,embedding_dim)
        # context_size개의 임베딩 벡터를 펼쳐서 은닉층으로 전달
        self.linear1 = nn.Linear(context_size * embedding_dim,hidden_dim)
        # 비선형 활성화 함수
        self.tanh = nn.Tanh()
        # 최종 출력 크기는 단어 사전 크기와 같다. 각 단어가 다음 단어일 점수를 출력한다.
        self.linear2 = nn.Linear(hidden_dim,vocab_size)

    def forward(self,x):
        x = self.embedding(x)           # x shape : (batch_size,context_size)
        # context 단어들의 임베딩을 하나의 긴 벡터로 펼친다.
        x = x.view(x.size(0),-1)        # x shape : (batch_size, context_size * embedding_dim)
        x = self.linear1(x)
        x = self.tanh(x)
        logits = self.linear2(x)        # logits shape : (batch_size, vocab_size)
        return logits

In [10]:
# 모델 객체 생성
model = NNLM(vocab_size,embedding_dim,context_size,hidden_dim)

# 손실 함수
criterion = nn.CrossEntropyLoss()

# 최적화 함수
optimizer = optim.SGD(model.parameters(),lr=0.1)

model

NNLM(
  (embedding): Embedding(7, 5)
  (linear1): Linear(in_features=10, out_features=10, bias=True)
  (tanh): Tanh()
  (linear2): Linear(in_features=10, out_features=7, bias=True)
)

In [12]:
# 학습
epochs = 300

for epoch in range(epochs):
    total_loss = 0

    for context, target in training_data:
        context_tensor = encode(context,word_to_index).unsqueeze(0)
        target_tensor = torch.tensor([word_to_index[target]],dtype=torch.long)

        optimizer.zero_grad()
        logits = model(context_tensor)
        loss = criterion(logits, target_tensor)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch+1) % 50 == 0:
        print(f'Epoch({epoch+1}/{epochs}) : Loss = {total_loss:4f}')

Epoch(50/300) : Loss = 4.177645
Epoch(100/300) : Loss = 4.031546
Epoch(150/300) : Loss = 4.001921
Epoch(200/300) : Loss = 3.987953
Epoch(250/300) : Loss = 3.977812
Epoch(300/300) : Loss = 3.960895


In [17]:
# 다음 단어 예측
def predict_next_word(context,sampling=False):
    """
    문맥 단어를 입력 받아 다음 단어를 예측하는 함수
    """

    input_tensor = encode(context, word_to_index).unsqueeze(0)

    model.eval()
    with torch.no_grad():
        logits = model(input_tensor)
        probs = F.softmax(logits,dim=1)

        if sampling:
            pred_id = torch.multinomial(probs,num_samples=1).item()
        else:
            # 확률이 가장 높은 단어 선택
            pred_id = torch.argmax(probs,dim=1).item()
    
    return index_to_word[pred_id], probs.squeeze()

next_word, probs = predict_next_word(['I','love'])
print('예측 단어 : ',next_word)
print('확률 :', probs)

예측 단어 :  NLP
확률 : tensor([1.2753e-03, 1.2121e-03, 2.6915e-01, 3.1690e-01, 3.9231e-04, 4.0988e-01,
        1.1952e-03])


In [18]:
# 단어별 예측 확률을 보기 쉽게 출력
context = ['I','love']
next_word, probs = predict_next_word(context)

for idx,prob in enumerate(probs):
    print(f'{index_to_word[idx]} : {prob.item():4f}')

I : 0.001275
love : 0.001212
you : 0.269151
deep : 0.316896
learning : 0.000392
NLP : 0.409879
<UNK> : 0.001195


In [20]:
# 간단한 문장 생성
def generate(seed_context, max_len=0,sampling = False):
    """
    NNLM으로 간단한 문자을 생성하는 함수
    """
    generated = list(seed_context)
    current_context = list(seed_context)

    for _ in range(max_len - len(seed_context)):
        next_word,_ = predict_next_word(current_context,sampling=sampling)
        generated.append(next_word)
        # 마지막 context_size 개 단어를 다음 입력 문맥으로 사용한다.
        current_context = generated[-context_size:]

    return ' '.join(generated)

print(generate(['I','love'],max_len = 8,sampling = False))
print(generate(['I','love'],max_len = 8,sampling = True))


I love NLP learning NLP learning NLP learning
I love you deep learning NLP learning NLP
